# How to compare and interpret models

Fitting one model is rarely the end of an analysis — the real question is usually *which of several candidate models best explains the data*. This page shows the HSSM pattern for that question:

1. fit each candidate with `idata_kwargs=dict(log_likelihood=True)` (comparison needs the pointwise log-likelihood),
2. rank the fits with [`az.compare`](https://python.arviz.org/en/stable/api/generated/arviz.compare.html), and
3. read the comparison table — including when it tells you the candidates are *indistinguishable*.

The worked example — two hierarchical regression models for drift rate — is adapted from the Winterbrain 2025 workshop ([archived snapshot](https://lnccbrown.github.io/HSSM/archive/hssm_tutorial_workshop_1/)). The stored outputs come from that run; the two hierarchical fits take a while if you re-execute them yourself.

## Run this how-to

<a href="https://colab.research.google.com/github/lnccbrown/HSSM/blob/main/docs/how_to/compare_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open this notebook in Google Colab"></a>

On Colab, uncomment and run the installation cell below once, then restart the runtime. For local setup, GPU extras, and troubleshooting see the [Installation guide](https://lnccbrown.github.io/HSSM/getting_started/installation/).


In [ ]:
# %pip install hssm

## Setup

In [1]:
# Import modules
import arviz as az
import jax
import matplotlib as plt
import pytensor

import hssm

pytensor.config.floatX = "float32"
jax.config.update("jax_enable_x64", False)

plt.use("Agg")
%matplotlib inline
%config InlineBackend.figure_format='retina'

# hssm.set_floatX("float32")

In [ ]:
import numpy as np
import pandas as pd

## The data: one outcome, two candidate explanations

We simulate a dataset in which each participant's drift rate depends on two (z-scored) brain-measure regressors (`zSTN`, `zGPe`) and a symptom-severity score (`sevScore`), with participants belonging to one of two diagnosis groups. The scientific question: **does diagnosis moderate the brain–behavior relationship (a fixed interaction), or is it enough to let effects vary across participants nested within diagnosis groups?**

In [27]:
# Function to simulate data for one participant
def simulate_participant3(participant_id, sevScore, diagnosis, size=300):
    """Simulate DDM trial data for one participant with diagnosis-dependent drift."""
    # intercept = 0.5
    zSTN = np.random.normal(loc=1, scale=2, size=size)
    zGPe = np.random.normal(loc=1, scale=2, size=size)

    if diagnosis == "PD":
        intercept = 0.3
    else:
        intercept = 0.6

    ## Strength of neural modulation depends on PD severity score
    if sevScore <= 0.2:
        v = intercept + 0.8 * zSTN + 0.3 * zGPe
    elif sevScore > 0.2 and sevScore <= 0.4:
        v = intercept + 0.6 * zSTN + 0.2 * zGPe
    elif sevScore > 0.4 and sevScore < 0.6:
        v = intercept + 0.5 * zSTN + 0.1 * zGPe
    else:
        v = intercept + 0.4 * zSTN + 0.0 * zGPe

    # Assume `hssm.simulate_data` returns a DataFrame
    true_values = np.column_stack(
        [v, np.repeat([[1.5, 0.5, 0.5]], axis=0, repeats=size)]
    )
    dataset_reg_v = hssm.simulate_data(
        model="ddm",
        theta=true_values,
        size=1,  # Generate one data point for each of the 1000 set of true values
    )

    # Adding additional variables to the dataset
    dataset_reg_v["zSTN"] = zSTN
    dataset_reg_v["zGPe"] = zGPe
    dataset_reg_v["participant_id"] = str(participant_id)
    dataset_reg_v["sevScore"] = sevScore
    dataset_reg_v["diagnosis"] = diagnosis

    return dataset_reg_v


# Simulate data for four participants
### note that we assume that STN & GPe

## PD patients:
# patients with low severity scores
subj_list = [1, 2, 3, 4, 5, 6, 7, 8, 1, 2, 3, 4, 5, 6, 7, 8]
sevscore_list = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
]
diagnosis_list = [
    "PD",
    "PD",
    "PD",
    "PD",
    "PD",
    "PD",
    "PD",
    "PD",
    "DD",
    "DD",
    "DD",
    "DD",
    "DD",
    "DD",
    "DD",
    "DD",
]

# [simulate_participant3(subj, sevscore, diagnosis) for subj, sevscore, diagnosis
# in zip(subj_list, sevscore_list, diagnosis_list)]
# dataset2_participant1 = simulate_participant3(1,0.10,"PD")
# dataset2_participant2 = simulate_participant3(2,0.20,"PD")
# dataset2_participant3 = simulate_participant3(3,0.30,"PD")
# dataset2_participant4 = simulate_participant3(4,0.40,"PD")
# # patients with high severity scores
# dataset2_participant5 = simulate_participant3(5,0.50,"PD")
# dataset2_participant6 = simulate_participant3(6,0.60,"PD")
# dataset2_participant7 = simulate_participant3(7,0.70,"PD")
# dataset2_participant8 = simulate_participant3(8,0.80,"PD")

# ## Dystonia patients:
# # patients with low severity scores
# dataset2_participant9 = simulate_participant3(1,0.10,"DD")
# dataset2_participant10 = simulate_participant3(2,0.20,"DD")
# dataset2_participant11 = simulate_participant3(3,0.30,"DD")
# dataset2_participant12 = simulate_participant3(4,0.40,"DD")
# # patients with high severity scores
# dataset2_participant13 = simulate_participant3(5,0.50,"DD")
# dataset2_participant14 = simulate_participant3(6,0.60,"DD")
# dataset2_participant15 = simulate_participant3(7,0.70,"DD")
# dataset2_participant16 = simulate_participant3(8,0.80,"DD")

# Combine datasets into one DataFrame
combined_dataset3 = pd.concat(
    [
        simulate_participant3(subj, sevscore, diagnosis)
        for subj, sevscore, diagnosis in zip(subj_list, sevscore_list, diagnosis_list)
    ],
    ignore_index=True,
)
combined_dataset3

,rt,response,zSTN,zGPe,participant_id,sevScore,diagnosis
0,0.936833,-1.0,-2.754070,-1.719626,1,0.1,PD
1,1.049654,1.0,2.736417,2.045129,1,0.1,PD
2,1.191599,1.0,-0.021384,0.377893,1,0.1,PD
3,2.432412,1.0,-0.380539,4.469055,1,0.1,PD
4,1.587661,1.0,1.653595,-0.396669,1,0.1,PD
...,...,...,...,...,...,...,...
4795,1.000964,1.0,0.393771,1.020571,8,0.8,DD
4796,2.113130,1.0,-1.310476,2.243242,8,0.8,DD
4797,1.511142,1.0,3.044100,0.773762,8,0.8,DD
4798,3.110676,1.0,-0.831975,-2.348503,8,0.8,DD


## Model 1: diagnosis as a fixed interaction

`v ~ 1 + (zSTN + zGPe)*sevScore*C(diagnosis) + ((1 + zSTN + zGPe)|participant_id)`

Note `idata_kwargs=dict(log_likelihood=True)` in the `sample()` call — without it, `az.compare` has nothing to work with.

In [28]:
model_reg_v_ex4_A1 = hssm.HSSM(
    data=combined_dataset3,
    include=[
        {
            "name": "v",
            "formula": (
                "v ~ 1 + (zSTN + zGPe)*sevScore*C(diagnosis) + "
                "((1 + zSTN + zGPe)|participant_id)"
            ),
            "prior": {
                # All ways to specify priors in the non-regression case
                # work the same way here.
                "Intercept": {"name": "Normal", "mu": 1.5, "sigma": 1.5},
                "zSTN": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "zGPe": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "sevScore": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "zSTN:sevScore": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "zGPe:sevScore": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "1|participant_id": {
                    "name": "Normal",
                    "mu": 0,
                    "sigma": {"name": "HalfNormal", "sigma": 1},
                },
                "zSTN|participant_id": {
                    "name": "Normal",
                    "mu": 0,
                    "sigma": {"name": "HalfNormal", "sigma": 1},
                },
                "zGPe|participant_id": {
                    "name": "Normal",
                    "mu": 0,
                    "sigma": {"name": "HalfNormal", "sigma": 1},
                },
            },
            "link": "identity",
        }
    ],
    noncentered=True,
    p_outlier=0.05,
)
model_reg_v_ex4_A1

Hierarchical Sequential Sampling Model
Model: ddm

Response variable: rt,response
Likelihood: analytical
Observations: 4800

Parameters:

v:
    Formula: v ~ 1 + (zSTN + zGPe)*sevScore*C(diagnosis) + ((1 + zSTN + zGPe)|participant_id)
    Priors:
        v_Intercept ~ Normal(mu: 1.5, sigma: 1.5)
        v_zSTN ~ Normal(mu: 0.0, sigma: 1.0)
        v_zGPe ~ Normal(mu: 0.0, sigma: 1.0)
        v_sevScore ~ Normal(mu: 0.0, sigma: 1.0)
        v_zSTN:sevScore ~ Normal(mu: 0.0, sigma: 1.0)
        v_zGPe:sevScore ~ Normal(mu: 0.0, sigma: 1.0)
        v_C(diagnosis) ~ Normal(mu: 0.0, sigma: 0.25)
        v_zSTN:C(diagnosis) ~ Normal(mu: 0.0, sigma: 0.25)
        v_zGPe:C(diagnosis) ~ Normal(mu: 0.0, sigma: 0.25)
        v_sevScore:C(diagnosis) ~ Normal(mu: 0.0, sigma: 0.25)
        v_zSTN:sevScore:C(diagnosis) ~ Normal(mu: 0.0, sigma: 0.25)
        v_zGPe:sevScore:C(diagnosis) ~ Normal(mu: 0.0, sigma: 0.25)
        v_1|participant_id ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 1.0))
        v

In [29]:
samples_model_reg_v_ex4_A1 = model_reg_v_ex4_A1.sample(
    sampler="numpyro",
    cores=3,
    chains=3,
    draws=500,
    tune=500,
    idata_kwargs=dict(log_likelihood=True),
)

In [30]:
az.summary(
    samples_model_reg_v_ex4_A1, var_names=["~_offset", "~_id"], filter_vars="like"
)  # var_names= ["~_offset"])

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
v_zSTN:sevScore:C(diagnosis)[PD],0.03,0.066,-0.072,0.14,1191,992,1.00,0.0019,0.0013
v_zSTN,0.864,0.068,0.76,0.97,550,703,1.01,0.003,0.0028
v_zGPe,0.388,0.04,0.33,0.45,482,497,1.00,0.0019,0.0021
v_sevScore,0.172,0.101,0.012,0.33,896,910,1.00,0.0034,0.0028
z,0.501,0.0063,0.49,0.51,1761,1118,1.00,0.00015,0.0001
v_zGPe:C(diagnosis)[PD],-0.042,0.03,-0.092,0.0042,732,899,1.00,0.0011,0.00079
v_sevScore:C(diagnosis)[PD],-0.272,0.113,-0.45,-0.099,919,977,1.00,0.0037,0.0027
v_zGPe:sevScore,-0.583,0.075,-0.7,-0.47,531,615,1.00,0.0033,0.0032
v_zSTN:C(diagnosis)[PD],-0.024,0.036,-0.083,0.032,1239,1177,1.00,0.001,0.0007
a,1.5001,0.0156,1.5,1.5,2215,1329,1.01,0.00033,0.00024


## Model 2: diagnosis as a grouping level

`v ~ 1 + (zSTN + zGPe)*sevScore + ((1 + zSTN + zGPe)|participant_id/C(diagnosis))`

Same data, same likelihood — the only difference is where diagnosis enters the model structure.

In [31]:
model_reg_v_ex4_A2 = hssm.HSSM(
    data=combined_dataset3,
    include=[
        {
            "name": "v",
            "formula": (
                "v ~ 1 + (zSTN + zGPe)*sevScore + "
                "((1 + zSTN + zGPe)|participant_id/C(diagnosis))"
            ),
            "prior": {
                # All ways to specify priors in the non-regression case
                # work the same way here.
                "Intercept": {"name": "Normal", "mu": 1.5, "sigma": 1.0},
                "zSTN": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "zGPe": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "sevScore": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "zSTN:sevScore": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "zGPe:sevScore": {"name": "Normal", "mu": 0, "sigma": 1.0},
                "1|participant_id": {
                    "name": "Normal",
                    "mu": 0,
                    "sigma": {"name": "HalfNormal", "sigma": 1},
                },
                "zSTN|participant_id": {
                    "name": "Normal",
                    "mu": 0,
                    "sigma": {"name": "HalfNormal", "sigma": 1},
                },
                "zGPe|participant_id": {
                    "name": "Normal",
                    "mu": 0,
                    "sigma": {"name": "HalfNormal", "sigma": 1},
                },
            },
            "link": "identity",
        }
    ],
    noncentered=True,
    p_outlier=0.05,
)

In [32]:
samples_model_reg_v_ex4_A2 = model_reg_v_ex4_A2.sample(
    sampler="numpyro",
    cores=3,
    chains=3,
    draws=500,
    tune=500,
    idata_kwargs=dict(log_likelihood=True),
)

In [33]:
az.summary(
    samples_model_reg_v_ex4_A2, var_names=["~_offset", "~_id"], filter_vars="like"
)

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
v_zGPe,0.363,0.034,0.31,0.42,858,758,1.00,0.0012,0.0011
v_zSTN,0.853,0.067,0.75,0.96,481,645,1.01,0.0032,0.0032
v_sevScore,0.04,0.27,-0.38,0.46,598,786,1.00,0.012,0.011
z,0.5012,0.0064,0.49,0.51,2245,1071,1.00,0.00013,0.0001
a,1.501,0.0154,1.5,1.5,2943,1032,1.01,0.00028,0.0002
v_zGPe:sevScore,-0.516,0.066,-0.62,-0.41,852,805,1.00,0.0023,0.0021
v_Intercept,0.46,0.14,0.25,0.69,627,721,1.00,0.0057,0.005
v_zSTN:sevScore,-0.63,0.13,-0.81,-0.42,538,664,1.00,0.0059,0.0065
t,0.5133,0.0059,0.5,0.52,1862,958,1.01,0.00014,9.6e-05


## Compare the models

`az.compare` ranks models by **expected log pointwise predictive density (ELPD)**, estimated via Pareto-smoothed importance-sampling leave-one-out cross-validation (LOO). Higher ELPD = better expected out-of-sample prediction.

In [34]:
az.compare(
    {"Model 1": samples_model_reg_v_ex4_A1, "Model 2": samples_model_reg_v_ex4_A2}
)

,rank,elpd_diff,dse,p_worse,diag_diff,diag_elpd,p,elpd,se,weight
Model 1,0,0.0,0.0,NaN,,,25.3,-5400.0,90.0,0.67
Model 2,1,-3.0,4.1,0.76,|elpd_diff| < 4,,36.4,-5400.0,90.0,0.33


## Reading the table

- **`rank` / `elpd`** — Model 1 ranks first, but rank alone is not a verdict.
- **`elpd_diff` vs. `dse`** — the difference to the best model (here ≈ 3.0) comes with a standard error (here ≈ 4.1). The difference is smaller than its own standard error, and ArviZ flags `|elpd_diff| < 4` as within noise: **the data do not distinguish these two models.** In that situation, prefer the simpler or more interpretable candidate rather than the nominal winner.
- **`weight`** — pseudo-Bayesian-model-averaging weights (here ≈ 0.67 / 0.33); useful for averaging predictions, not a posterior probability that a model is "true".
- **`p`** — the effective number of parameters; a sanity check that a model is not vastly more flexible than its competitor for the same predictive payoff.

Two practical cautions: LOO estimates can be unreliable when Pareto-$k$ diagnostics are high (ArviZ warns when this happens), and ELPD compares *predictive* performance — a model can predict well while its parameters remain hard to interpret (or vice versa).

## See also

- [The HSSM tutorial](https://lnccbrown.github.io/HSSM/tutorials/main_tutorial/) — section on validating and comparing models
- [ArviZ model-comparison guide](https://python.arviz.org/en/stable/user_guide/index.html)
- [A complete scientific workflow](https://lnccbrown.github.io/HSSM/tutorials/scientific_workflow_hssm/) — model comparison embedded in a full analysis